# AFML real-graph experiments on Colab

This notebook runs the existing Arxiv/temporal/MAG workflow on an explicitly selected GPU. It does not claim CUDA has already been tested on the laptop. Run the backend checks before long jobs. Free Colab GPU availability and runtime duration are not guaranteed: https://research.google.com/colaboratory/faq.html

1. Export the current local code using `python examples/export_colab_bundle.py --output runs/real-research-colab.zip`.
2. Upload that ZIP in the next cell. It contains the explicit source/config/test/notebook allowlist, no local datasets or arbitrary runtime folders.
3. Choose Runtime → Change runtime type → GPU. Keep the notebook interactive; this does not use SSH or circumvent runtime limits.

The GPU run uses fresh source banks and keeps its evidence separate from the CPU study. It may take multiple sessions. Preserve completed artifacts; do not silently lower epochs, seeds or candidate counts to fit a runtime limit.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, json, hashlib
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Choose one explicit exported project ZIP")
archive_path = Path(next(iter(uploaded)))
PROJECT = Path('/content/amortized-gnn-tta')
if PROJECT.exists():
    raise FileExistsError('Project directory exists; inspect it before another extraction')
with zipfile.ZipFile(archive_path) as archive:
    for entry in archive.infolist():
        path=Path(entry.filename)
        if path.is_absolute() or '..' in path.parts:
            raise ValueError('Invalid archive member')
    archive.extractall('/content')
manifest=json.loads((PROJECT/'colab_bundle.json').read_text())
for name,expected in manifest['files_sha256'].items():
    if hashlib.sha256((PROJECT/name).read_bytes()).hexdigest()!=expected:
        raise ValueError('Exported bundle checksum mismatch: '+name)
print('Verified',len(manifest['files_sha256']),'exported files')


In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable,'-m','pip','install','-e',str(PROJECT)+'[dev,ogb,perf]'],check=True)
os.chdir(PROJECT)
sys.path.insert(0,str(PROJECT/'src'))
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA runtime available; select a GPU runtime before proceeding')
print(torch.__version__,torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi'],check=True)


## Check software and the actual backend

The ordinary tests check behavior on small CPU fixtures. The next probe additionally exercises all three GNNs and both generated update types on the selected GPU, including five-candidate selection and CPU-compatible checkpoint restoration. Random graphs here are software checks, not research results.


In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q'],check=True)
subprocess.run([sys.executable,'examples/check_adaptation_device.py','--device','cuda','--output-dir','runs/colab-device-probe'],check=True)


In [ ]:
sys.path.insert(0,str(PROJECT/'examples'))
from run_arxiv_experiments import create_generator
from amortized_gnn_tta.models import make_model_spec,build_model
from amortized_gnn_tta.adaptation.benchmark import generated_predictions
from amortized_gnn_tta.adaptation.candidates import agreement_predictions
from amortized_gnn_tta.adaptation.checkpoints import save_generator_checkpoint,load_generator_checkpoint
from torch_geometric.data import Data
probe_plan=dict(lora_rank=2,diffusion_steps=7,sampling_steps=3)
rng=torch.Generator().manual_seed(31)
graph=Data(x=torch.randn(200,8,generator=rng),edge_index=torch.stack((torch.arange(199),torch.arange(1,200))),num_nodes=200)
results=[]
for architecture in ('gcn','gat','graphsage'):
    model=build_model(make_model_spec(architecture,8,40,hidden_channels=8)).cuda().eval()
    for update in ('head','lora'):
        for method in ('hypernetwork','diffusion'):
            descriptor,generator,schema,scope=create_generator(model,'D',update,method,probe_plan)
            descriptor=descriptor.cuda();generator=generator.cuda().eval()
            def predict(g,d):
                if method=='diffusion':
                    return agreement_predictions(model,graph,d,g,indices=torch.arange(50),scope=scope,device='cuda')[0]
                return generated_predictions(model,graph,d,g,scope=scope,device='cuda')
            output=predict(generator,descriptor)
            path=PROJECT/'runs/colab-generator-probe'/f'{architecture}-{update}-{method}.pt'
            save_generator_checkpoint(path,generator,descriptor,schema,target_scope=scope,provenance={'purpose':'random-model CUDA check, not accuracy evidence'})
            restored,d,_=load_generator_checkpoint(path,model=model,device='cuda')
            replay=predict(restored,d)
            torch.testing.assert_close(output.logits,replay.logits,rtol=1e-5,atol=1e-6)
            results.append(dict(architecture=architecture,update=update,method=method,finite=bool(torch.isfinite(output.logits).all()),checkpoint_replay=True))
(PROJECT/'runs/colab-generator-probe/RESULTS.json').write_text(json.dumps(results,indent=2))
print('CUDA generator checks:',len(results))


## Download official public data, then preserve the protocol

Downloads go to this runtime's dataset directory. Arxiv uses the official archive. Pair-Align country files are verified against published checksums. No credentials or Drive mount are needed. Dataset licenses/attribution are described in the project decisions.

The three configurations retain their seeds, source budgets, epochs, corruptions and candidate policy. Only the execution device changes. CPU and GPU numerical equality is not promised.


In [ ]:
from amortized_gnn_tta.data.ogb import load_arxiv
from amortized_gnn_tta.data.natural import load_country,COUNTRY_MD5
OGB_ROOT=Path('/content/afml-data/ogb');MAG_ROOT=Path('/content/afml-data/mag-country')
load_arxiv(OGB_ROOT,download=True)
for country in COUNTRY_MD5:
    load_country(MAG_ROOT,country,download=True)
for name in ('arxiv_research_v1.json','arxiv_temporal_research_v1.json','mag_country_research_v1.json'):
    path=PROJECT/'configs'/name
    plan=json.loads(path.read_text());plan['device']='cuda'
    path.write_text(json.dumps(plan,indent=2)+chr(10))


## Run the sequential experiments

This creates fresh directories and refuses accidental overwrite. It trains the controlled source bank, runs Experiments 1–4, trains separate temporal/country banks and generators for Experiment 5, then runs development diagnostics for Experiment 6. All final label scoring follows frozen validation choices. Inspect progress in the printed stage names and per-stage logs.

Do not treat CPU sampled RSS as GPU memory usage. GPU memory per method requires a separate measured probe before making a GPU memory-efficiency claim. Long-run GPU correctness/performance is unverified until this notebook is actually executed successfully.


In [ ]:
run_id='colab-real-research-v1'
BANK=PROJECT/'runs'/f'{run_id}-controlled-bank'
STUDY=PROJECT/'runs'/run_id
subprocess.run([sys.executable,'examples/run_real_research_pipeline.py',
    '--bank',str(BANK),'--ogb-root',str(OGB_ROOT),'--mag-root',str(MAG_ROOT),'--output-dir',str(STUDY)],check=True)


## Preserve evidence

The ZIP below contains only this study and its explicitly named controlled source bank, including checkpoints, exact snapshots, configurations, predictions, deltas, raw timing and reports. Download it before the runtime expires. It may be large. It must not be committed to Git. Partial runs remain partial: inspect `JOURNAL.json` and each bank status instead of assuming completion from the existence of a folder.


In [ ]:
evidence=Path('/content')/(run_id+'-evidence.zip')
with zipfile.ZipFile(evidence,'x',compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in (BANK,STUDY,PROJECT/'runs/colab-device-probe',PROJECT/'runs/colab-generator-probe'):
        if folder.exists():
            for path in sorted(folder.rglob('*')):
                if path.is_file():archive.write(path,path.relative_to(PROJECT))
print('Evidence bytes:',evidence.stat().st_size)
files.download(str(evidence))
